In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn statsmodels joblib

# Social Media Post Recommender

**Luna's Project — Lighthouse Sanctuary Philippines**

This notebook builds a recommendation engine that answers:
> *What should Lighthouse post, on which platform, on which day, and at what time — to maximize both followers and donation value?*

**Dataset:** 812 social media posts across 7 platforms (Facebook, Instagram, Twitter, WhatsApp, TikTok, LinkedIn, YouTube).

**Two optimization targets:**
- `estimated_donation_value_php` — maximize fundraising
- `engagement_rate` — maximize follower growth

**Outputs:**
1. Best platform per goal
2. Best post type per goal
3. Best day of week per goal
4. Best time of day (hour) per goal
5. Best content topic per goal
6. Recommended posting frequency per platform
7. Gradient Boosting model — predict donation value for any post configuration
8. Final recommendation card (saved as `recommendations.json` for frontend)

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
from pathlib import Path

# Resolve seed CSV whether the kernel cwd is this folder or the repo root
_cwd = Path.cwd().resolve()
_candidates = [
    _cwd.parent.parent / "Backend" / "Data" / "SeedData" / "social_media_posts.csv",
    _cwd / "Backend" / "Data" / "SeedData" / "social_media_posts.csv",
]
DATA_PATH = None
for anchor in _candidates[:2]:
    if anchor.exists():
        DATA_PATH = anchor
        break
if DATA_PATH is None:
    smr = _cwd / "MachineLearning" / "SocialMediaRecomendation"
    if smr.is_dir():
        p = smr.parent.parent / "Backend" / "Data" / "SeedData" / "social_media_posts.csv"
        if p.exists():
            DATA_PATH = p
if DATA_PATH is None:
    raise FileNotFoundError(
        "social_media_posts.csv not found. Open/run Jupyter with cwd = MachineLearning/SocialMediaRecomendation "
        "or the IntexIIWednesday repo root."
    )

df = pd.read_csv(DATA_PATH)

# Clean types
df['estimated_donation_value_php'] = pd.to_numeric(df['estimated_donation_value_php'], errors='coerce')
df['engagement_rate'] = pd.to_numeric(df['engagement_rate'], errors='coerce')
df['follower_count_at_post'] = pd.to_numeric(df['follower_count_at_post'], errors='coerce')
df['boost_budget_php'] = pd.to_numeric(df['boost_budget_php'], errors='coerce').fillna(0)
df['is_boosted'] = df['is_boosted'].astype(str).str.lower().map({'true': True, 'false': False, '1': True, '0': False}).fillna(False)
df['has_call_to_action'] = df['has_call_to_action'].astype(str).str.lower().map({'true': True, 'false': False, '1': True, '0': False}).fillna(False)
df['features_resident_story'] = df['features_resident_story'].astype(str).str.lower().map({'true': True, 'false': False, '1': True, '0': False}).fillna(False)
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')

print(f'Loaded {len(df)} posts across {df["platform"].nunique()} platforms')
print(f'Date range: {df["created_at"].min().date()} to {df["created_at"].max().date()}')
df.head(3)

## 2. Platform Analysis

Which platform performs best for **donations** vs **engagement**?

In [ ]:
platform_stats = df.groupby('platform').agg(
    posts=('post_id', 'count'),
    avg_donation=('estimated_donation_value_php', 'mean'),
    median_donation=('estimated_donation_value_php', 'median'),
    avg_engagement=('engagement_rate', 'mean'),
    avg_followers=('follower_count_at_post', 'mean'),
    total_donation_referrals=('donation_referrals', 'sum'),
).round(2).sort_values('avg_donation', ascending=False)

print('=== PLATFORM PERFORMANCE ===')
print(platform_stats.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
platform_stats['avg_donation'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Avg Estimated Donation Value by Platform (PHP)')
axes[0].set_xlabel('PHP')
platform_stats['avg_engagement'].sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Avg Engagement Rate by Platform')
plt.tight_layout()
plt.savefig('platform_performance.png', dpi=120, bbox_inches='tight')
plt.show()

best_donation_platform = platform_stats['avg_donation'].idxmax()
best_engagement_platform = platform_stats['avg_engagement'].idxmax()
print(f'\nBest platform for DONATIONS:  {best_donation_platform} (avg PHP {platform_stats.loc[best_donation_platform, "avg_donation"]:,.0f})')
print(f'Best platform for ENGAGEMENT: {best_engagement_platform}')

## 3. Post Type Analysis

In [ ]:
type_stats = df.groupby('post_type').agg(
    posts=('post_id', 'count'),
    avg_donation=('estimated_donation_value_php', 'mean'),
    avg_engagement=('engagement_rate', 'mean'),
).round(2).sort_values('avg_donation', ascending=False)

print('=== POST TYPE PERFORMANCE ===')
print(type_stats.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
type_stats['avg_donation'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Avg Donation Value by Post Type (PHP)')
type_stats['avg_engagement'].sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Avg Engagement Rate by Post Type')
plt.tight_layout()
plt.savefig('post_type_performance.png', dpi=120, bbox_inches='tight')
plt.show()

best_donation_type = type_stats['avg_donation'].idxmax()
best_engagement_type = type_stats['avg_engagement'].idxmax()
print(f'\nBest post type for DONATIONS:  {best_donation_type} (avg PHP {type_stats.loc[best_donation_type, "avg_donation"]:,.0f})')
print(f'Best post type for ENGAGEMENT: {best_engagement_type}')

## 4. Day of Week & Time of Day Analysis

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

day_stats = df.groupby('day_of_week').agg(
    avg_donation=('estimated_donation_value_php', 'mean'),
    avg_engagement=('engagement_rate', 'mean'),
    posts=('post_id', 'count')
).reindex(day_order).round(2)

hour_stats = df.groupby('post_hour').agg(
    avg_donation=('estimated_donation_value_php', 'mean'),
    avg_engagement=('engagement_rate', 'mean'),
    posts=('post_id', 'count')
).round(2)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
day_stats['avg_donation'].plot(kind='bar', ax=axes[0,0], color='steelblue', rot=30)
axes[0,0].set_title('Avg Donation Value by Day of Week (PHP)')
day_stats['avg_engagement'].plot(kind='bar', ax=axes[0,1], color='coral', rot=30)
axes[0,1].set_title('Avg Engagement Rate by Day of Week')
hour_stats['avg_donation'].plot(kind='line', ax=axes[1,0], color='steelblue', marker='o', markersize=4)
axes[1,0].set_title('Avg Donation Value by Hour of Day (PHP)')
axes[1,0].set_xlabel('Hour (0-23)')
axes[1,0].set_xticks(range(0, 24, 2))
hour_stats['avg_engagement'].plot(kind='line', ax=axes[1,1], color='coral', marker='o', markersize=4)
axes[1,1].set_title('Avg Engagement Rate by Hour of Day')
axes[1,1].set_xlabel('Hour (0-23)')
axes[1,1].set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.savefig('timing_performance.png', dpi=120, bbox_inches='tight')
plt.show()

best_day_donation = day_stats['avg_donation'].idxmax()
best_day_engagement = day_stats['avg_engagement'].idxmax()
best_hour_donation = int(hour_stats['avg_donation'].idxmax())
best_hour_engagement = int(hour_stats['avg_engagement'].idxmax())
print(f'Best day for DONATIONS:   {best_day_donation} (avg PHP {day_stats.loc[best_day_donation, "avg_donation"]:,.0f})')
print(f'Best day for ENGAGEMENT:  {best_day_engagement}')
print(f'Best hour for DONATIONS:  {best_hour_donation:02d}:00 (avg PHP {hour_stats.loc[best_hour_donation, "avg_donation"]:,.0f})')
print(f'Best hour for ENGAGEMENT: {best_hour_engagement:02d}:00')

## 5. Content Topic & Sentiment Analysis

In [ ]:
topic_stats = df.groupby('content_topic').agg(
    avg_donation=('estimated_donation_value_php', 'mean'),
    avg_engagement=('engagement_rate', 'mean'),
    posts=('post_id', 'count')
).round(2).sort_values('avg_donation', ascending=False)

tone_stats = df.groupby('sentiment_tone').agg(
    avg_donation=('estimated_donation_value_php', 'mean'),
    avg_engagement=('engagement_rate', 'mean'),
    posts=('post_id', 'count')
).round(2).sort_values('avg_donation', ascending=False)

print('=== CONTENT TOPIC PERFORMANCE ==='); print(topic_stats.to_string())
print(); print('=== SENTIMENT TONE PERFORMANCE ==='); print(tone_stats.to_string())

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
topic_stats['avg_donation'].sort_values().plot(kind='barh', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Avg Donation Value by Content Topic (PHP)')
topic_stats['avg_engagement'].sort_values().plot(kind='barh', ax=axes[0,1], color='coral')
axes[0,1].set_title('Avg Engagement Rate by Content Topic')
tone_stats['avg_donation'].sort_values().plot(kind='barh', ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Avg Donation Value by Sentiment Tone (PHP)')
tone_stats['avg_engagement'].sort_values().plot(kind='barh', ax=axes[1,1], color='coral')
axes[1,1].set_title('Avg Engagement Rate by Sentiment Tone')
plt.tight_layout()
plt.savefig('content_performance.png', dpi=120, bbox_inches='tight')
plt.show()

best_topic_donation = topic_stats['avg_donation'].idxmax()
best_topic_engagement = topic_stats['avg_engagement'].idxmax()
best_tone_donation = tone_stats['avg_donation'].idxmax()
best_tone_engagement = tone_stats['avg_engagement'].idxmax()
print(f'Best topic for DONATIONS:  {best_topic_donation}')
print(f'Best topic for ENGAGEMENT: {best_topic_engagement}')
print(f'Best tone for DONATIONS:   {best_tone_donation}')
print(f'Best tone for ENGAGEMENT:  {best_tone_engagement}')

## 6. Posting Frequency Analysis

How often should Lighthouse post on each platform?

In [ ]:
df['week'] = df['created_at'].dt.to_period('W')
weekly_freq = df.groupby(['platform', 'week']).size().reset_index(name='posts_per_week')
freq_stats = weekly_freq.groupby('platform')['posts_per_week'].agg(['mean', 'median', 'max']).round(2)
freq_stats.columns = ['avg_posts_per_week', 'median_posts_per_week', 'max_posts_per_week']
freq_stats = freq_stats.join(platform_stats[['avg_donation', 'avg_engagement']])
freq_stats = freq_stats.sort_values('avg_donation', ascending=False)

print('=== POSTING FREQUENCY BY PLATFORM ===')
print(freq_stats.to_string())

platform_weekly = df.groupby(['platform', 'week']).agg(
    posts=('post_id', 'count'),
    avg_donation=('estimated_donation_value_php', 'mean')
).reset_index()
corr = platform_weekly.groupby('platform').apply(lambda x: x['posts'].corr(x['avg_donation'])).round(3)
print()
print('=== Correlation: Posts Per Week vs Avg Donation Value ===')
print('(Positive = more posts -> more donations on that platform)')
print(corr.sort_values(ascending=False).to_string())

## 7. Predictive Model — Gradient Boosting

Train a model to predict `estimated_donation_value_php` from post configuration.
This lets us score any hypothetical post before publishing.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

model_df = df[[
    'platform', 'post_type', 'day_of_week', 'post_hour',
    'content_topic', 'sentiment_tone', 'media_type',
    'has_call_to_action', 'is_boosted', 'features_resident_story',
    'num_hashtags', 'caption_length', 'boost_budget_php',
    'estimated_donation_value_php'
]].dropna().copy()

cat_cols = ['platform', 'post_type', 'day_of_week', 'content_topic', 'sentiment_tone', 'media_type']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    encoders[col] = le

model_df['has_call_to_action'] = model_df['has_call_to_action'].astype(int)
model_df['is_boosted'] = model_df['is_boosted'].astype(int)
model_df['features_resident_story'] = model_df['features_resident_story'].astype(int)

feature_cols = [
    'platform', 'post_type', 'day_of_week', 'post_hour',
    'content_topic', 'sentiment_tone', 'media_type',
    'has_call_to_action', 'is_boosted', 'features_resident_story',
    'num_hashtags', 'caption_length', 'boost_budget_php'
]

X = model_df[feature_cols]
y = model_df['estimated_donation_value_php']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gb = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
gb.fit(X_train, y_train)

y_pred = gb.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
cv_r2 = cross_val_score(gb, X, y, cv=5, scoring='r2').mean()

print(f'Test MAE:       PHP {mae:,.0f}')
print(f'Test R2:        {r2:.3f}')
print(f'CV R2 (5-fold): {cv_r2:.3f}')

joblib.dump({'model': gb, 'encoders': encoders, 'feature_cols': feature_cols}, 'recommender_model.pkl')
print('\nModel saved to recommender_model.pkl')

## 8. Feature Importance

In [ ]:
importances = pd.Series(gb.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importance — Donation Value Predictor')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('Top 5 features driving donation value:')
print(importances.tail(5).sort_values(ascending=False).to_string())

## 9. Per-Platform Best Time Heatmap

In [ ]:
pivot_donation = df.pivot_table(
    values='estimated_donation_value_php',
    index='platform',
    columns='post_hour',
    aggfunc='mean'
)

plt.figure(figsize=(18, 5))
sns.heatmap(pivot_donation, cmap='YlOrRd', linewidths=0.3,
            cbar_kws={'label': 'Avg Donation Value (PHP)'})
plt.title('Best Posting Hour by Platform — Donation Value (PHP)')
plt.xlabel('Hour of Day')
plt.ylabel('Platform')
plt.tight_layout()
plt.savefig('platform_hour_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print('=== BEST HOUR TO POST (per platform, donation-optimized) ===')
platform_best_hours = {}
for platform in pivot_donation.index:
    best_h = pivot_donation.loc[platform].idxmax()
    val = pivot_donation.loc[platform, best_h]
    if not pd.isna(val):
        platform_best_hours[platform] = int(best_h)
        print(f'  {platform:<12} -> {int(best_h):02d}:00  (avg PHP {val:,.0f})')

## 10. Recommender Engine & Final Output Card

In [ ]:
import json

def get_top_n(series, n=3):
    return series.nlargest(n).index.tolist()

def build_recommendation(goal='donations'):
    metric = 'avg_donation' if goal == 'donations' else 'avg_engagement'
    rec = {
        'goal': goal,
        'platform': platform_stats[metric].idxmax(),
        'post_type': type_stats[metric].idxmax(),
        'day_of_week': day_stats[metric].idxmax(),
        'best_hour': int(hour_stats[metric].idxmax()),
        'content_topic': topic_stats[metric].idxmax(),
        'sentiment_tone': tone_stats[metric].idxmax(),
        'top_3_platforms': get_top_n(platform_stats[metric]),
        'top_3_post_types': get_top_n(type_stats[metric]),
        'top_3_topics': get_top_n(topic_stats[metric]),
        'recommended_posts_per_week': round(freq_stats.loc[platform_stats[metric].idxmax(), 'avg_posts_per_week']),
    }
    boosted_val = df[df['is_boosted'] == True]['estimated_donation_value_php' if goal == 'donations' else 'engagement_rate'].mean()
    unboosted_val = df[df['is_boosted'] == False]['estimated_donation_value_php' if goal == 'donations' else 'engagement_rate'].mean()
    rec['boost_recommended'] = bool(boosted_val > unboosted_val)
    rec['boost_lift_pct'] = round(((boosted_val - unboosted_val) / unboosted_val) * 100, 1) if unboosted_val else 0
    return rec

rec_donations = build_recommendation('donations')
rec_engagement = build_recommendation('engagement')

rec_card = {
    'donation_strategy': {
        'primary_platform': rec_donations['platform'],
        'top_platforms': rec_donations['top_3_platforms'],
        'post_type': rec_donations['post_type'],
        'top_post_types': rec_donations['top_3_post_types'],
        'best_day': rec_donations['day_of_week'],
        'best_hour_global': f"{rec_donations['best_hour']:02d}:00",
        'best_hour_per_platform': {p: f"{h:02d}:00" for p, h in platform_best_hours.items()},
        'content_topic': rec_donations['content_topic'],
        'top_topics': rec_donations['top_3_topics'],
        'sentiment_tone': rec_donations['sentiment_tone'],
        'use_resident_story': True,
        'include_cta': True,
        'boost_post': rec_donations['boost_recommended'],
        'boost_lift_pct': rec_donations['boost_lift_pct'],
        'posts_per_week': rec_donations['recommended_posts_per_week'],
    },
    'engagement_strategy': {
        'primary_platform': rec_engagement['platform'],
        'top_platforms': rec_engagement['top_3_platforms'],
        'post_type': rec_engagement['post_type'],
        'best_day': rec_engagement['day_of_week'],
        'best_hour_global': f"{rec_engagement['best_hour']:02d}:00",
        'content_topic': rec_engagement['content_topic'],
        'sentiment_tone': rec_engagement['sentiment_tone'],
        'posts_per_week': rec_engagement['recommended_posts_per_week'],
    },
    'model_performance': {
        'algorithm': 'GradientBoostingRegressor',
        'test_mae_php': round(mae),
        'test_r2': round(r2, 3),
        'cv_r2_5fold': round(cv_r2, 3),
        'training_samples': len(X_train),
    }
}

with open('recommendations.json', 'w') as f:
    json.dump(rec_card, f, indent=2)

print('=== FINAL RECOMMENDATION CARD ===')
print(json.dumps(rec_card, indent=2))
print('\nSaved to recommendations.json')

## Summary

Results from running this pipeline on Lighthouse Philippines social media data (812 posts, 7 platforms).

`recommendations.json` is saved in this folder and consumed by the frontend recommendation view.